In [ ]:
import os
import sys
from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt

# Add src to path for editable installs (optional)
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    src_path = repo_root / "src"
    if (src_path / "jorg").exists():
        sys.path.insert(0, str(src_path))
        data_dir = repo_root / "data"
        if data_dir.exists():
            os.environ.setdefault("JORG_DATA_DIR", str(data_dir))
        break
    repo_root = repo_root.parent


---
## 1. Basic Synthesis with Jorg

The standard `synth()` function is the main entry point. It computes a stellar spectrum given:
- **Teff**: Effective temperature [K]
- **logg**: Surface gravity (log g)
- **m_H**: Metallicity [M/H]

In [ ]:
from jorg.synthesis import synth
from jorg.lines.linelist_data import get_VALD_solar_linelist

# Load the built-in VALD solar linelist
linelist = get_VALD_solar_linelist()
print(f"Loaded {len(linelist)} spectral lines")

# Solar parameters
Teff = 5780  # K
logg = 4.44
m_H = 0.0    # Solar metallicity

# Synthesize a small wavelength region
t0 = time.perf_counter()
wl, flux, cont = synth(
    Teff, logg, m_H,
    wavelengths=(5160, 5162),  # small Mg I region
    linelist=linelist,
    rectify=False,
    hydrogen_lines=False,
    verbose=False,
)
t1 = time.perf_counter()

print(f"Synthesis time: {t1-t0:.2f}s")
print(f"Wavelength points: {len(wl)}")

# Plot (physical flux + continuum)
plt.figure(figsize=(10, 4), dpi=120)
plt.plot(wl, flux, "b-", lw=0.8, label="Flux")
plt.plot(wl, cont, "r--", lw=0.8, alpha=0.7, label="Continuum")
plt.xlabel("Wavelength [Å]")
plt.ylabel("Flux")
plt.title(f"Solar Spectrum (Teff={Teff}K, logg={logg}, [M/H]={m_H})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 2. Vectorized Radiative Transfer

The core optimization in Jorg is **vectorized radiative transfer** using JAX. This replaces nested Python loops with a single fused kernel.

### Key function: `compute_I_linear_batch(tau, source, mu_values)`

- Computes intensity for all wavelengths × all μ angles simultaneously
- Achieves orders-of-magnitude speedup once JIT compiled

**Important**: end-to-end synthesis time is often dominated by chemical equilibrium (per-layer) unless you reuse chemical equilibrium across repeated runs or use a fully vectorized CE path.


In [ ]:
import jax.numpy as jnp
from jorg.radiative_transfer_exact import (
    compute_I_linear_batch,
    generate_mu_grid,
    compute_I_linear_mu,
)

# Create test data: 56 atmospheric layers, 1000 wavelength points
n_layers = 56
n_wavelengths = 1000

# Random optical depth (increasing with layer depth)
np.random.seed(42)
tau_matrix = jnp.cumsum(jnp.abs(np.random.randn(n_layers, n_wavelengths) * 0.1), axis=0)
source_matrix = jnp.ones((n_layers, n_wavelengths)) * 1.0  # Uniform source

# Angular quadrature grid
mu_values, mu_weights = generate_mu_grid(20)
print(f"mu grid: {len(mu_values)} points from {mu_values.min():.3f} to {mu_values.max():.3f}")

# Warm up JIT compilation
_ = compute_I_linear_batch(tau_matrix, source_matrix, mu_values).block_until_ready()

# Benchmark vectorized version
t0 = time.perf_counter()
for _ in range(10):
    intensity = compute_I_linear_batch(tau_matrix, source_matrix, mu_values).block_until_ready()
t_vectorized = (time.perf_counter() - t0) / 10

print(f"\nVectorized RT:")
print(f"  Output shape: {intensity.shape}  (n_mu, n_layers, n_wavelengths)")
print(f"  Time per call: {t_vectorized*1000:.2f} ms")
print(f"  Throughput: {n_wavelengths * len(mu_values) / t_vectorized:.0f} (wl × μ)/s")

In [ ]:
# Plot all spectra
fig, axes = plt.subplots(2, 3, figsize=(12, 6), dpi=120, sharex=True, sharey=True)
axes = axes.ravel()

for i, (ax, Teff) in enumerate(zip(axes, Teffs)):
    # Rectified flux = flux / continuum
    rect = result.flux[i] / result.continuum[i]
    ax.plot(result.wavelengths, rect, 'b-', lw=0.8)
    ax.set_title(f'Teff = {Teff:.0f} K', fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.grid(alpha=0.3)

for ax in axes[-3:]:
    ax.set_xlabel('Wavelength [Å]')
for ax in axes[::3]:
    ax.set_ylabel('Rectified Flux')

fig.suptitle('Temperature Sequence (dwarf stars)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Plot the chunked result
plt.figure(figsize=(12, 4), dpi=120)
plt.plot(wl_chunked, flux_chunked / cont_chunked, 'b-', lw=0.5)
plt.xlabel('Wavelength [Å]')
plt.ylabel('Rectified Flux')
plt.title('Solar Spectrum (5150-5200 Å) - Chunked Synthesis')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. GPU Memory Management

Jorg provides utilities to estimate memory usage and optimize chunk/batch sizes.

---
## 8. Custom Abundances

You can specify individual element abundances relative to solar values.

In [ ]:
# Compare solar vs enhanced magnesium
wl_solar, flux_solar, cont_solar = synth(
    5780, 4.44, 0.0,
    wavelengths=(5165, 5190),
    linelist=linelist,
    rectify=False,
    hydrogen_lines=False,
    verbose=False,
)

# Enhanced Mg (+0.3 dex)
wl_mg, flux_mg, cont_mg = synth(
    5780, 4.44, 0.0,
    wavelengths=(5165, 5190),
    linelist=linelist,
    Mg=0.3,  # [Mg/H] = +0.3
    rectify=False,
    hydrogen_lines=False,
    verbose=False,
)

plt.figure(figsize=(10, 5), dpi=120)
plt.plot(wl_solar, flux_solar / cont_solar, "b-", lw=0.8, label="Solar [Mg/H]=0.0")
plt.plot(wl_mg, flux_mg / cont_mg, "r-", lw=0.8, alpha=0.8, label="Enhanced [Mg/H]=+0.3")
plt.xlabel("Wavelength [Å]")
plt.ylabel("Rectified Flux")
plt.title("Mg I Triplet Region: Solar vs Mg-Enhanced")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
## 9. Comparison with Korg.jl

Jorg is a Python reimplementation of [Korg.jl](https://github.com/ajwheeler/Korg.jl), a Julia-based spectral synthesis code. This section compares Jorg's output with pre-computed Korg.jl reference spectra to validate accuracy.

### Reference Data

We provide Korg.jl reference spectra for three stellar types:
- **Solar-like**: Teff=5771K, logg=4.44, [M/H]=0.0
- **Arcturus-like**: Teff=4250K, logg=1.4, [M/H]=-0.5
- **Metal-poor K Giant**: Teff=4500K, logg=1.5, [M/H]=-2.5

All computed for 5000-5020 Å with the same VALD linelist.

In [ ]:
# Load Korg.jl reference data
def load_korg_reference(filename):
    """Load Korg.jl reference spectrum from text file."""
    korg_ref_dir = repo_root / 'examples' / 'korg_reference'
    filepath = korg_ref_dir / filename
    
    # Parse header for parameters
    params = {}
    with open(filepath, 'r') as f:
        for line in f:
            if not line.startswith('#'):
                break
            if 'Parameters:' in line:
                # Parse "Parameters: Teff=5771.0K, logg=4.44, [M/H]=0.0"
                parts = line.split('Parameters:')[1].strip()
                for p in parts.split(','):
                    p = p.strip()
                    if 'Teff=' in p:
                        params['Teff'] = float(p.split('=')[1].rstrip('K'))
                    elif 'logg=' in p:
                        params['logg'] = float(p.split('=')[1])
                    elif '[M/H]=' in p:
                        params['m_H'] = float(p.split('=')[1])
            elif 'Synthesis time:' in line:
                params['korg_time'] = float(line.split(':')[1].strip().split()[0])
    
    # Load data (skip header lines starting with #)
    data = np.loadtxt(filepath, comments='#')
    return {
        'wavelength': data[:, 0],
        'flux': data[:, 1],
        'continuum': data[:, 2],
        'normalized': data[:, 3],
        **params
    }

# Load all reference spectra
korg_solar = load_korg_reference('korg_solar_with_lines.txt')
korg_arcturus = load_korg_reference('korg_arcturus_with_lines.txt')
korg_metal_poor = load_korg_reference('korg_metal_poor_k_giant_with_lines.txt')

print("Loaded Korg.jl reference spectra:")
for name, ref in [('Solar', korg_solar), ('Arcturus', korg_arcturus), ('Metal-poor K Giant', korg_metal_poor)]:
    print(f"  {name}: Teff={ref['Teff']:.0f}K, logg={ref['logg']:.2f}, [M/H]={ref['m_H']:.1f}")
    print(f"    Korg.jl synthesis time: {ref['korg_time']:.2f}s")

In [ ]:
# Synthesize with Jorg and compare to Korg.jl
#
# The reference files are 5000-5020 Å, but for tutorial runtime we compare
# a smaller subset by default. Set COMPARE_WL_RANGE=None to use the full range.
COMPARE_WL_RANGE = (5000.0, 5002.0)

def compare_with_korg(korg_ref, linelist, wl_subset=COMPARE_WL_RANGE, verbose=True):
    """Run Jorg synthesis and compare with a Korg.jl reference spectrum."""
    Teff = korg_ref["Teff"]
    logg = korg_ref["logg"]
    m_H = korg_ref["m_H"]

    wl_korg_full = korg_ref["wavelength"]
    norm_korg_full = korg_ref["normalized"]

    if wl_subset is None:
        mask = np.ones_like(wl_korg_full, dtype=bool)
        wl_range = (float(wl_korg_full.min()), float(wl_korg_full.max()))
    else:
        wl_min, wl_max = wl_subset
        mask = (wl_korg_full >= wl_min) & (wl_korg_full <= wl_max)
        wl_range = (float(wl_korg_full[mask].min()), float(wl_korg_full[mask].max()))

    wl_korg = wl_korg_full[mask]
    norm_korg = norm_korg_full[mask]

    # Run Jorg synthesis
    t0 = time.perf_counter()
    wl_jorg, flux_jorg, cont_jorg = synth(
        Teff, logg, m_H,
        wavelengths=wl_range,
        linelist=linelist,
        rectify=False,
        hydrogen_lines=False,
        verbose=False,
    )
    jorg_time = time.perf_counter() - t0

    # Normalized flux
    norm_jorg = flux_jorg / cont_jorg

    # Interpolate Jorg to the (subset) Korg wavelength grid
    norm_jorg_interp = np.interp(wl_korg, wl_jorg, norm_jorg)

    # Compute metrics
    residual = norm_jorg_interp - norm_korg
    mae = np.mean(np.abs(residual))
    rms = np.sqrt(np.mean(residual**2))
    max_err = np.max(np.abs(residual))
    correlation = np.corrcoef(norm_jorg_interp, norm_korg)[0, 1]
    agreement_1pct = 100 * np.mean(np.abs(residual) < 0.01)

    if verbose:
        print(f"Teff={Teff:.0f}K, logg={logg:.2f}, [M/H]={m_H:.1f}")
        print(f"  Range: {wl_range[0]:.2f}-{wl_range[1]:.2f} Å")
        # print(f"  Jorg time: {jorg_time:.2f}s | Korg.jl time (full file): {korg_ref["korg_time"]:.2f}s")
        print(f"  MAE: {mae:.6f} | RMS: {rms:.6f} | Max: {max_err:.6f}")
        print(f"  Corr: {correlation:.6f} | Within 1%: {agreement_1pct:.1f}%")

    return {
        "wl_korg": wl_korg,
        "norm_korg": norm_korg,
        "wl_jorg": wl_jorg,
        "norm_jorg": norm_jorg,
        "norm_jorg_interp": norm_jorg_interp,
        "residual": residual,
        "mae": mae,
        "rms": rms,
        "max_err": max_err,
        "correlation": correlation,
        "agreement_1pct": agreement_1pct,
        "jorg_time": jorg_time,
        "korg_time": korg_ref["korg_time"],
        "wl_range": wl_range,
    }


print("=" * 60)
label = "full (5000-5020 Å)" if COMPARE_WL_RANGE is None else f"{COMPARE_WL_RANGE[0]:.0f}-{COMPARE_WL_RANGE[1]:.0f} Å"
print(f"Jorg vs Korg.jl Comparison ({label})")
print("=" * 60)
print()

results = {}
for name, ref in [("Solar", korg_solar), ("Arcturus", korg_arcturus), ("Metal-poor", korg_metal_poor)]:
    print(f"--- {name} ---")
    results[name] = compare_with_korg(ref, linelist)
    print()


In [ ]:
# Plot comparison: Jorg vs Korg.jl
fig, axes = plt.subplots(3, 2, figsize=(14, 10), dpi=120)

refs = [("Solar", korg_solar), ("Arcturus", korg_arcturus), ("Metal-poor", korg_metal_poor)]

for i, (name, ref) in enumerate(refs):
    res = results[name]
    wl = res["wl_korg"]
    norm_korg = res["norm_korg"]

    # Left panel: spectra overlay
    ax1 = axes[i, 0]
    ax1.plot(wl, norm_korg, "b-", lw=0.8, label="Korg.jl", alpha=0.8)
    ax1.plot(wl, res["norm_jorg_interp"], "r--", lw=0.8, label="Jorg", alpha=0.8)
    ax1.set_ylabel("Normalized Flux")
    ax1.set_title(f"{name}: Teff={ref["Teff"]:.0f}K, logg={ref["logg"]:.2f}, [M/H]={ref["m_H"]:.1f}")
    ax1.legend(loc="lower right", fontsize=8)
    ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1.1)

    # Right panel: residuals
    ax2 = axes[i, 1]
    ax2.plot(wl, res["residual"] * 100, "k-", lw=0.5)
    ax2.axhline(0, color="gray", linestyle="--", lw=0.5)
    ax2.axhline(1, color="red", linestyle=":", lw=0.5, alpha=0.5)
    ax2.axhline(-1, color="red", linestyle=":", lw=0.5, alpha=0.5)
    ax2.set_ylabel("Residual (%)")
    ax2.set_title(f"MAE={res["mae"]*100:.3f}%, RMS={res["rms"]*100:.3f}%")
    ax2.grid(alpha=0.3)
    ax2.set_ylim(-3, 3)

# Set x-labels only on bottom row
for ax in axes[-1, :]:
    ax.set_xlabel("Wavelength [Å]")

plt.tight_layout()
plt.savefig("jorg_korg_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: jorg_korg_comparison.png")


### Speed Comparison

This section summarizes accuracy vs Korg.jl and gives rough timing for Jorg.

**Important**:
- If your JAX backend is `cpu` (no CUDA-enabled `jaxlib`), you will not see GPU acceleration.
- Jorg timings include Python overhead and any one-time caching/JIT effects.
- To keep this tutorial runnable, we compare a small wavelength subset by default.


In [ ]:
# Summary table: Speed and Accuracy
print("=" * 75)
print("                    Jorg vs Korg.jl: Speed & Accuracy Summary")
print("=" * 75)
label = "full" if COMPARE_WL_RANGE is None else f"{COMPARE_WL_RANGE[0]:.0f}-{COMPARE_WL_RANGE[1]:.0f} Å"
print(f"Comparison range: {label} (Korg files are 5000-5020 Å)")
print()
print(f"{"Star Type":<18} {"Jorg (s)":<10} {"Korg.jl (s)":<12} {"MAE (%)":<10} {"Agreement":<10}")
print("-" * 75)

for name in ["Solar", "Arcturus", "Metal-poor"]:
    res = results[name]
    print(f"{name:<18} {res["jorg_time"]:>8.2f}   {res["korg_time"]:>10.2f}   "
          f"{res["mae"]*100:>8.4f}   {res["agreement_1pct"]:>8.1f}%")

print("-" * 75)
print()

# Calculate average metrics
avg_mae = np.mean([results[n]["mae"] for n in results]) * 100
avg_agreement = np.mean([results[n]["agreement_1pct"] for n in results])
total_jorg = sum([results[n]["jorg_time"] for n in results])
total_korg = sum([results[n]["korg_time"] for n in results])

print(f"Average MAE: {avg_mae:.4f}%")
print(f"Average Agreement (within 1%): {avg_agreement:.1f}%")
print(f"Total Jorg time: {total_jorg:.2f}s | Total Korg.jl time: {total_korg:.2f}s")
print()
print("Note: Jorg uses caching/JIT for some kernels; first run can be slower than subsequent runs.")


In [ ]:
# Benchmark after warmup: repeated runs
print("Jorg Speed After Warmup (Solar parameters)")
print("-" * 50)

wl_range = (5000.0, 5020.0) if COMPARE_WL_RANGE is None else COMPARE_WL_RANGE
print(f"Range: {wl_range[0]:.2f}-{wl_range[1]:.2f} Å")
print()

times = []
for i in range(3):
    t0 = time.perf_counter()
    wl, flux, cont = synth(
        korg_solar["Teff"], korg_solar["logg"], korg_solar["m_H"],
        wavelengths=wl_range,
        linelist=linelist,
        rectify=False,
        hydrogen_lines=False,
        verbose=False,
    )
    t = time.perf_counter() - t0
    times.append(t)
    print(f"  Run {i+1}: {t:.2f}s")

print("-" * 50)
print(f"  Mean: {np.mean(times):.2f}s ± {np.std(times):.2f}s")
print(f"  Korg.jl reference (full file): {korg_solar["korg_time"]:.2f}s")
print()


In [ ]:
print("Tutorial complete!")